# Solar Filament Segmentation 2026 — U-Net baseline and label-noise analysis

This notebook reproduces the submission and every number in the report from the scripts in `src/`. It expects the competition data under `data/MAGFiLO_1.0_Kaggle_2026/` (or set `FILAMENT_DATA`). GPU required for training; set `TRAIN = False` to only evaluate existing checkpoints in `checkpoints/`.

In [ ]:
import os, json, subprocess, sys
from pathlib import Path
os.environ['FILAMENT_SCALE'] = '1'          # full 2048 x 2048 resolution
TRAIN = True
TAGS = ['v3_full_crop1024', 'v3_full_crop1024_s2']
def run(cmd):
    print('$', cmd); subprocess.run(cmd, shell=True, check=True)
print('data:', os.environ.get('FILAMENT_DATA', 'data/MAGFiLO_1.0_Kaggle_2026'))

## 1. Data, cache and the fixed split
707 H-alpha images carry 1,154 annotation sets (296 images were annotated by 2-3 people). The split is by capture date (the first 8 characters of the file name) so that the same Sun seen by several observatories never sits on both sides: 603 train / 104 validation files, seed 20261020.

In [ ]:
run(f'{sys.executable} src/unet.py cache')
split = json.load(open('src/split_seed20261020.json'))
print(split['unit'], '| train', len(split['train']), 'val', len(split['val']), 'val dates', len(split['val_dates']))

## 2. Training (two seeds, one change per run)
ResNet-34 encoder (ImageNet init, 1-channel input), U-Net decoder, BCE + Dice, AdamW 3e-4 cosine, flips / rot90 / intensity augmentation, bf16 autocast, random 1024 x 1024 crops of the full-resolution image, 100 epochs. The checkpoint with the best validation PQ (merged annotation sets) is kept.

In [ ]:
if TRAIN:
    run(f'{sys.executable} src/unet.py train --tag {TAGS[0]} --epochs 100 --bs 4 --crop 1024 --min-area 160')
    run(f'{sys.executable} src/unet.py train --tag {TAGS[1]} --epochs 100 --bs 4 --crop 1024 --min-area 160 --seed 20261021')
for t in TAGS:
    print(t, 'exists:', Path(f'checkpoints/unet_{t}_best.pt').exists())

## 3. Post-processing tuned on the validation split
Instances are the connected components of the averaged probability map above a threshold; fragments closer than 2 x merge_r pixels are merged; components smaller than min_area are dropped. The sweep changes one factor at a time and selects on the merged-GT PQ, which is what the public leaderboard tracks.

In [ ]:
run(f"{sys.executable} src/postproc.py probs --tags {','.join(TAGS)}")
run(f"{sys.executable} src/postproc.py sweep --tags {','.join(TAGS)} --thr 0.7 --min-area 320")
best = json.load(open(f"results_post_{'+'.join(TAGS)}_s1.json"))['best']
print('best on val:', best)

## 4. What the judges score: PQ = SQ x RQ, IoU / Dice distributions, one-to-many relations, and the human ceiling
`evaluate.py` scores the configuration against (a) the merged annotation sets, (b) each annotation set separately, and (c) annotator 2 against annotator 1 on the validation images that were annotated twice.

In [ ]:
run(f"{sys.executable} src/evaluate.py --tags {','.join(TAGS)} --thr {best['thr']} --min-area {best['min_area']} --merge-r {best['merge_r']} --min-mean {best['min_mean']} --limb {best['limb']}")
ev = json.load(open('results_eval.json'))
for k in ('merged', 'per_set', 'inter_annotator'):
    r = ev[k]
    if r: print(f"{k:16s} PQ {r['PQ']:.3f} SQ {r['SQ']:.3f} RQ {r['RQ']:.3f}  TP {r['TP']} FP {r['FP']} FN {r['FN']}  IoU p50 {r['iou_percentiles']['50']:.3f}  Dice p50 {r['dice_percentiles']['50']:.3f}")

## 5. Submission
One row per predicted filament, `filament_id = <image stem>_<n>`, COCO compressed RLE at 2048 x 2048.

In [ ]:
run(f"{sys.executable} src/postproc.py predict --tags {','.join(TAGS)} --thr {best['thr']} --min-area {best['min_area']} --merge-r {best['merge_r']} --min-mean {best['min_mean']} --limb {best['limb']}")
import pandas as pd, glob
f = sorted(glob.glob('submissions/submission_post_*.csv'), key=os.path.getmtime)[-1]
d = pd.read_csv(f); print(f, len(d), 'filaments over', d.filament_id.str.rsplit('_', n=1).str[0].nunique(), 'images')

## 6. Ablation (validation PQ, merged annotation sets, tuned post-processing; public LB where submitted)

| variant (one change each) | val PQ | public LB |
|---|---|---|
| training-free threshold baseline | 0.12 | 0.12 |
| U-Net v1: ResNet-34, 1/2 resolution, 40 epochs | 0.346 | 0.34 |
| + 100 epochs | 0.350 | |
| + second seed, probabilities averaged | 0.356 | |
| full resolution, 1024 crops, 100 epochs (v3) | 0.359 | 0.35 |
| v3 second seed / two seeds averaged | 0.360 / 0.360 | 0.35 |

Measured to not help: limb/disk-edge mask, per-component mean-probability filter, mask dilation or erosion by 1 px (-0.06 to -0.10), hysteresis thresholding, duplicating predictions. Inter-annotator PQ on the same validation images: 0.330.